# 04 — Blinded Error-Taxonomy Annotation and Cohen’s Kappa


> **Post-shared-task analysis.** Gold test labels were public when this analysis was
> designed. Nothing in this notebook changes the official CI=0.035 submission or the
> third-place ranking. Results are retrospective and must not be described as untouched
> test-set estimates.

This notebook prepares two independent, blinded annotation sheets for the 35
errors of QLoRA 2,600 and computes raw agreement and Cohen's kappa after both
raters finish. Do not expose either completed sheet to the other rater before
annotation. A full 35-item second pass is preferred; set `IAA_N=15` only if
resources require a predeclared subsample.

**Labels**

- `function_intent_event`: same main entities, different purpose/function/activity.
- `visual_material`: directly observable colour/texture/material/shape/script cue.
- `recognition`: different object, place, food, instrument, or scene identity.

In [ ]:
from pathlib import Path
from collections import Counter
import csv, io, json, os, urllib.request, zipfile

import numpy as np
import pandas as pd


def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'README.md').exists() and (candidate / 'Test').exists():
            return candidate.resolve()
    raise FileNotFoundError('Run this notebook from the IE2026-HalDetect repository.')


ROOT = find_repo_root()
HERE = ROOT / 'post_task_analysis'
CACHE = HERE / 'cache'
OUTPUT = HERE / 'outputs'
CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

GOLD_URL = (
    'https://huggingface.co/datasets/QCRI/ImageEval-ArabicNLP26/'
    'resolve/main/task1b/test_en.jsonl'
)
gold_override = os.getenv('IMAGEEVAL_TEST_GOLD')
GOLD_PATH = Path(gold_override) if gold_override else CACHE / 'test_en.jsonl'
if not GOLD_PATH.exists():
    print('Downloading released gold test JSONL...')
    urllib.request.urlretrieve(GOLD_URL, GOLD_PATH)


def read_gold(path=GOLD_PATH):
    rows = [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()]
    assert len(rows) == 1000, f'Expected 1,000 gold items, found {len(rows)}'
    frame = pd.DataFrame(rows)
    assert frame['id'].is_unique
    assert frame['labels'].map(lambda x: len(x) == 3 and sum(x) == 1).all()
    frame['gold_idx'] = frame['labels'].map(lambda x: x.index(True))
    return frame


def load_prediction_zip(path, gold):
    path = Path(path)
    assert path.exists(), path
    with zipfile.ZipFile(path) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith('.csv')]
        assert len(csv_names) == 1, (path, csv_names)
        with io.TextIOWrapper(archive.open(csv_names[0]), encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))
    raw = pd.DataFrame(rows)
    required = {'id', 'statement_index', 'prediction'}
    assert required.issubset(raw.columns), (path, raw.columns)
    raw['statement_index'] = raw['statement_index'].astype(int)
    raw['pred_bool'] = raw['prediction'].str.strip().str.lower().map(
        {'true': True, 'false': False})
    assert raw['pred_bool'].notna().all(), f'Unparseable prediction in {path}'
    assert not raw.duplicated(['id', 'statement_index']).any()
    assert set(raw['statement_index']) == {0, 1, 2}
    grouped = raw.sort_values(['id', 'statement_index']).groupby('id', sort=False)
    vectors = grouped['pred_bool'].apply(list)
    assert vectors.map(len).eq(3).all()
    assert set(vectors.index) == set(gold['id']), f'ID mismatch in {path}'

    result = gold[['id', 'gold_idx']].copy()
    by_id = vectors.to_dict()
    result['pred_vector'] = result['id'].map(by_id)
    result['format_valid'] = result['pred_vector'].map(lambda x: sum(x) == 1)
    result['pred_idx'] = result['pred_vector'].map(
        lambda x: x.index(True) if sum(x) == 1 else np.nan)
    result['correct'] = result['format_valid'] & result['pred_idx'].eq(result['gold_idx'])
    result['error'] = ~result['correct']
    return result


SUBMISSIONS = {
    'Elimination': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-elimination/prediction_en.zip',
    'Socratic': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-socratic/prediction_en.zip',
    'Devils advocate': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-devils-advocate/prediction_en.zip',
    'Evidence first': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-evidence-first/prediction_en.zip',
    'Attribute checklist': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-attribute-checklist/prediction_en.zip',
    'Confidence ranked': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-confidence-ranked/prediction_en.zip',
    'QLoRA 2,000': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2k-image/prediction_en.zip',
    'QLoRA 2,348': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p3k-image/prediction_en.zip',
    'QLoRA 2,600': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p6k-image/prediction_en.zip',
    'QLoRA 3,000 legacy': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-3k-image/prediction_en.zip',
}

gold = read_gold()
predictions = {name: load_prediction_zip(path, gold) for name, path in SUBMISSIONS.items()}
summary = pd.DataFrame([
    {
        'system': name,
        'n': len(frame),
        'errors': int(frame['error'].sum()),
        'CI': frame['error'].mean(),
        'accuracy': frame['correct'].mean(),
        'format_failures': int((~frame['format_valid']).sum()),
    }
    for name, frame in predictions.items()
]).sort_values(['CI', 'system']).reset_index(drop=True)
display(summary)

## Create blinded sheets (safe: existing sheets are never overwritten)

In [ ]:
ANNOTATION_DIR = HERE / 'annotations'
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
IAA_N = 35
IAA_SEED = 2026
SYSTEM = 'QLoRA 2,600'

evaluated = gold.merge(
    predictions[SYSTEM][['id', 'pred_idx', 'correct', 'error']], on='id')
errors = evaluated.loc[evaluated['error']].copy()
assert len(errors) == 35
if IAA_N < len(errors):
    selected_ids = errors.sample(IAA_N, random_state=IAA_SEED)['id']
    errors = errors[errors['id'].isin(selected_ids)].copy()

errors['statement_1'] = errors['statements'].str[0]
errors['statement_2'] = errors['statements'].str[1]
errors['statement_3'] = errors['statements'].str[2]
errors['image_url'] = errors['image'].map(
    lambda path: 'https://huggingface.co/datasets/QCRI/ImageEval-ArabicNLP26/resolve/main/' + path)
columns = [
    'id', 'image_url', 'country', 'category', 'subcategory',
    'statement_1', 'statement_2', 'statement_3',
]
for rater, seed in [('rater1', IAA_SEED + 1), ('rater2', IAA_SEED + 2)]:
    path = ANNOTATION_DIR / f'{rater}.csv'
    if path.exists():
        print('Preserving existing sheet:', path)
        continue
    sheet = errors[columns].sample(frac=1, random_state=seed).copy()
    sheet['label'] = ''
    sheet['notes'] = ''
    sheet.to_csv(path, index=False)
    print('Created:', path)

## Optional item viewer

In [ ]:
from IPython.display import Image, Markdown, display


def show_annotation_item(rater='rater2', row_number=0):
    sheet = pd.read_csv(ANNOTATION_DIR / f'{rater}.csv', keep_default_na=False)
    row = sheet.iloc[row_number]
    display(Markdown(
        f"**{row_number + 1}/{len(sheet)} — {row['country']} | "
        f"{row['category']} | {row['subcategory']}**"
    ))
    display(Image(url=row['image_url'], width=600))
    for index in range(1, 4):
        display(Markdown(f"**Statement {index}:** {row[f'statement_{index}']}"))


show_annotation_item('rater2', 0)

## Agreement and Cohen’s kappa

In [ ]:
LABELS = {'function_intent_event', 'visual_material', 'recognition'}


def load_completed_rater(name):
    path = ANNOTATION_DIR / f'{name}.csv'
    frame = pd.read_csv(path, keep_default_na=False)
    frame['label'] = frame['label'].str.strip()
    blanks = frame['label'].eq('').sum()
    unknown = sorted(set(frame['label']) - LABELS - {''})
    if blanks or unknown:
        raise ValueError(f'{name}: blanks={blanks}, unknown={unknown}')
    assert frame['id'].is_unique
    return frame[['id', 'label']].rename(columns={'label': name})


try:
    rater1 = load_completed_rater('rater1')
    rater2 = load_completed_rater('rater2')
except ValueError as exc:
    print('Complete both sheets, then rerun this cell:', exc)
else:
    paired = rater1.merge(rater2, on='id', validate='one_to_one')
    assert len(paired) == IAA_N
    observed = (paired['rater1'] == paired['rater2']).mean()
    p1 = paired['rater1'].value_counts(normalize=True)
    p2 = paired['rater2'].value_counts(normalize=True)
    expected = sum(p1.get(label, 0) * p2.get(label, 0) for label in LABELS)
    kappa = (observed - expected) / (1 - expected) if expected < 1 else 1.0
    summary = {
        'n_double_annotated': len(paired),
        'raw_agreement': observed,
        'cohen_kappa': kappa,
        'n_disagreements': int((paired['rater1'] != paired['rater2']).sum()),
    }
    print(json.dumps(summary, indent=2))
    confusion = pd.crosstab(paired['rater1'], paired['rater2'])
    display(confusion)
    disagreements = paired.loc[paired['rater1'] != paired['rater2']]
    disagreements.to_csv(ANNOTATION_DIR / 'disagreements_for_adjudication.csv', index=False)
    (ANNOTATION_DIR / 'iaa_summary.json').write_text(
        json.dumps(summary, indent=2), encoding='utf-8')

## Reporting template

“A second annotator independently labelled **N** errors using a pre-specified
three-class rubric. Raw agreement was **X** and Cohen's $\kappa$ was **Y**.
Disagreements were [adjudicated by discussion / retained without adjudication].”